# IndoFraud-Synth & Skew-Aware FL — Kaggle Runner (T4×2 / P100, 30 GB RAM)

Kaggle is ideal for running the full experimental matrix for free:
- **30 GB RAM** (handles 6M-row `IndoFraud-Synth` and wide `IEEE-CIS` without OOM)
- **GPU `T4 x2` or `P100`** (30 hrs/week free) — with `T4 x2`, you can run **two seed shards simultaneously** on GPU 0 and GPU 1!
- **Built-in public benchmarks:** Attach ULB, IEEE-CIS, or Sparkov directly via **+ Add Input** (no uploading!)
- **`uv`** installs the missing packages in ~15s while keeping Kaggle's CUDA PyTorch intact

### Notebook Settings (Right Sidebar on Kaggle)
1. **Accelerator:** `GPU T4 x2` (recommended) or `GPU P100`
2. **Internet:** `ON` (needed to install `uv`, `flwr`, `catboost` and/or clone GitHub)
3. *(Optional)* **Add Input:**
   - `mlg-ulb/creditcardfraud` (ULB)
   - `ieee-fraud-detection` (IEEE-CIS)
   - `kartik2112/fraud-detection` (Sparkov)
   - Your previous notebook version's Output (to resume cleanly with `--skip-existing`)

In [ ]:
# 1. Inspect Kaggle hardware & install uv
import os, shutil

!nvidia-smi || true
!echo "CPUs: $(nproc)" && free -h

if shutil.which("uv") is None:
    !curl -LsSf https://astral.sh/uv/install.sh | sh
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

!uv --version

In [ ]:
# 2. Clone or copy repository + install dependencies via uv
import os

# Option A: Clone from GitHub
REPO_URL = ""  # e.g., "https://github.com/<your-user>/IndoFraudSynth.git"
# Option B: Copy from an uploaded Kaggle Dataset (used if REPO_URL is empty)
KAGGLE_CODE_INPUT = "/kaggle/input/indofraud-synth-code"

WORK_DIR = "/kaggle/working/IndoFraudSynth"
if REPO_URL:
    if not os.path.exists(WORK_DIR):
        !git clone {REPO_URL} {WORK_DIR}
    else:
        !git -C {WORK_DIR} pull --ff-only
else:
    assert os.path.isdir(KAGGLE_CODE_INPUT), (
        f"Set REPO_URL or upload the repo as a Kaggle dataset at {KAGGLE_CODE_INPUT}."
    )
    !cp -r "{KAGGLE_CODE_INPUT}" "{WORK_DIR}"

%cd {WORK_DIR}

# Fast uv install (only installs missing packages; preserves Kaggle's CUDA PyTorch & XGBoost)
import importlib.util, subprocess
need = []
for mod, pkg in [
    ("catboost", "catboost>=1.2.7"),
    ("flwr", "flwr>=1.13.0"),
    ("xgboost", "xgboost>=2.1.1"),
    ("pyarrow", "pyarrow>=15.0"),
    ("yaml", "pyyaml"),
    ("pytest", "pytest"),
]:
    if importlib.util.find_spec(mod) is None:
        need.append(pkg)
if need:
    print("Installing missing packages with uv:", need)
    subprocess.run(
        ["uv", "pip", "install", "--system", "--only-binary=:all:", *need],
        check=True,
    )
else:
    print("All required packages already installed.")

# Cache Parquet versions of read-only /kaggle/input CSVs into writable /kaggle/working
os.environ["INDOFRAUD_DATA_CACHE"] = "/kaggle/working/_data_cache"
os.makedirs(os.environ["INDOFRAUD_DATA_CACHE"], exist_ok=True)

import torch, xgboost as xgb, catboost
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
print(f"XGBoost {xgb.__version__} | CatBoost {catboost.__version__} | CPUs: {os.cpu_count()}")

In [ ]:
# 3. Auto-detect attached Kaggle inputs & optionally resume from a previous run
import glob, os, shutil

RESULTS_ROOT = "/kaggle/working/results"
os.makedirs(RESULTS_ROOT, exist_ok=True)

# Optional: set to previous notebook output dir under /kaggle/input/ to resume via --skip-existing
PREV_RESULTS_INPUT = ""  # e.g., "/kaggle/input/indofraud-run-v1/results"
if PREV_RESULTS_INPUT and os.path.isdir(PREV_RESULTS_INPUT):
    print(f"Copying previous results from {PREV_RESULTS_INPUT} -> {RESULTS_ROOT} ...")
    shutil.copytree(PREV_RESULTS_INPUT, RESULTS_ROOT, dirs_exist_ok=True)

# Auto-locate public datasets attached via Kaggle '+ Add Input'
def find_first(patterns):
    for pat in patterns:
        hits = glob.glob(pat)
        if hits:
            return hits[0]
    return None

PATHS = {
    "ulb": find_first(["/kaggle/input/*/creditcard.csv", "/kaggle/input/*/*/creditcard.csv"]),
    "ieee_cis": os.path.dirname(p) if (p := find_first(["/kaggle/input/*/train_transaction.csv"])) else None,
    "sparkov": os.path.dirname(p) if (p := find_first(["/kaggle/input/*/fraudTrain.csv"])) else None,
    "indofraud": os.path.dirname(p) if (p := find_first(["/kaggle/input/*/transactions.csv.gz"])) else None,
}
print("Detected datasets:", PATHS)

# If IndoFraud-Synth wasn't attached as an Input dataset, generate it into /kaggle/working/data/full
if PATHS["indofraud"] is None:
    INDO_DATA = "/kaggle/working/data/full"
    if not os.path.exists(f"{INDO_DATA}/transactions.csv.gz"):
        !python -m indofraud_synth --config configs/default.yaml --out "{INDO_DATA}"
    PATHS["indofraud"] = INDO_DATA
print("IndoFraud-Synth path:", PATHS["indofraud"])

In [ ]:
# 4A. Single-GPU Suite Run (works on P100 or T4)
# Choose DATASET in {"indofraud", "ulb", "ieee_cis", "sparkov"}
DATASET = "indofraud"
DATA_DIR = PATHS[DATASET]
assert DATA_DIR is not None, f"Attach the Kaggle input dataset for {DATASET} first!"

CONFIG = "configs/default.yaml"
PARTITIONS = "profile" if DATASET == "indofraud" else "dirichlet:0.5"
CONDITIONS = "baseline,normalized,normalized_fedprox,normalized_fedbn,normalized_size_fedbn"
SEEDS = "1,2,3,4,5"
TRAIN_CAP = 200000      # 0 = uncapped (fits in 30 GB RAM on Kaggle); 200000 is ~2x faster
PATIENCE = 3
MLP_BATCH_SIZE = 4096
OUT_DIR = f"{RESULTS_ROOT}/suite_{DATASET}"

!python -m fl_experiments.run_suite \
    --config {CONFIG} \
    --dataset {DATASET} \
    --data "{DATA_DIR}" \
    --engine local \
    --device auto \
    --train-cap {TRAIN_CAP} \
    --patience {PATIENCE} \
    --mlp-batch-size {MLP_BATCH_SIZE} \
    --partitions "{PARTITIONS}" \
    --conditions "{CONDITIONS}" \
    --seeds "{SEEDS}" \
    --out "{OUT_DIR}" \
    --skip-existing

In [ ]:
# 4B. BONUS — Dual-GPU Parallel Sharding (for Kaggle 'GPU T4 x2' accelerator)
# Splits 5 seeds across GPU 0 (seeds 1,2,3) and GPU 1 (seeds 4,5) concurrently (~2x speedup)!
# Pre-warm Parquet cache before spawning parallel workers so both processes read Parquet cleanly.
import os, subprocess, torch

RUN_DUAL_GPU = False  # Set to True when using Kaggle 'GPU T4 x2' instead of Cell 4A

if RUN_DUAL_GPU:
    assert torch.cuda.device_count() >= 2, "Select Accelerator = GPU T4 x2 in Kaggle settings."
    # Pre-warm Parquet cache once
    from fl_experiments.data import load_indofraud, load_ulb, load_ieee_cis, load_sparkov
    _loaders = {"indofraud": load_indofraud, "ulb": load_ulb, "ieee_cis": load_ieee_cis, "sparkov": load_sparkov}
    _ = _loaders[DATASET](DATA_DIR)
    del _

    n_cpus = os.cpu_count() or 4
    threads_per_gpu = max(1, n_cpus // 2)

    def spawn_shard(gpu_id: int, seeds: str):
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        cmd = [
            "python", "-m", "fl_experiments.run_suite",
            "--config", CONFIG, "--dataset", DATASET, "--data", DATA_DIR,
            "--engine", "local", "--device", "cuda", "--threads", str(threads_per_gpu),
            "--train-cap", str(TRAIN_CAP), "--patience", str(PATIENCE),
            "--mlp-batch-size", str(MLP_BATCH_SIZE),
            "--partitions", PARTITIONS, "--conditions", CONDITIONS,
            "--seeds", seeds, "--out", OUT_DIR, "--skip-existing",
        ]
        log = open(f"/kaggle/working/gpu{gpu_id}.log", "w")
        return subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log

    p0, l0 = spawn_shard(0, "1,2,3")
    p1, l1 = spawn_shard(1, "4,5")
    rc0, rc1 = p0.wait(), p1.wait()
    l0.close(); l1.close()
    print(f"GPU 0 exit={rc0} | GPU 1 exit={rc1}")
    assert rc0 == 0 and rc1 == 0, "Check /kaggle/working/gpu0.log and gpu1.log"

In [ ]:
# 5. Compute final Friedman-Nemenyi stats, inspect outputs, & bundle results for download
import os
from IPython.display import display, Markdown, Image

part_tag = PARTITIONS.split(",")[0].replace(":", "_a").replace(".", "p")
SUITE_PART_DIR = f"{OUT_DIR}/{part_tag}"

!python -m fl_experiments.stats --suite-dir "{SUITE_PART_DIR}" --metric pooled_ap

summary_md = os.path.join(SUITE_PART_DIR, "summary.md")
cd_png = os.path.join(SUITE_PART_DIR, "cd_diagram.png")
if os.path.exists(summary_md):
    display(Markdown(open(summary_md).read()))
if os.path.exists(cd_png):
    display(Image(filename=cd_png))

# Package lightweight results archive (excludes large _model_cache binaries)
!tar --exclude='_model_cache' -czf /kaggle/working/results_bundle.tar.gz -C /kaggle/working results
!ls -lh /kaggle/working/results_bundle.tar.gz